# 03. 추론 · 서빙 템플릿 — vLLM / SGLang / TensorRT

| # | 기법 | 섹션 |
|---|---|---|
| 20 | vLLM OpenAI 호환 API (스트리밍, 동시 요청) | 1–2 |
| 21 | Speculative Decoding | 3 |
| 22 · 23 | FlashInfer · AWQ/GPTQ 4bit | 3 (모드 비교) |
| 24 | Structured Generation (JSON 스키마 · 선택지 · 도구 호출) | 4 |
| 8 | 노트북 안에서 vLLM 직접 실행 (`vLLM (venv)` 커널) | 5 |
| 34 | Torch-TensorRT (비LLM, CNN 추론 컴파일) | 6 |
| 35 | ONNX Runtime + TensorRT EP | 7 |

**서빙 엔진 전환은 서버 관리자 명령**(호스트)으로 합니다. 노트북 코드는 그대로 둔 채 엔진만 바꿔 비교하세요.
```bash
sudo bash scripts/17_serve.sh vllm        # 기본 (AWQ)
sudo bash scripts/17_serve.sh vllm-spec   # + Speculative Decoding
sudo bash scripts/17_serve.sh vllm-fi     # + FlashInfer
sudo bash scripts/17_serve.sh vllm-gptq   # GPTQ 비교
sudo bash scripts/17_serve.sh vllm-big    # 14B AWQ
sudo bash scripts/17_serve.sh sglang      # SGLang
sudo bash scripts/17_serve.sh stop        # 종료 (6·7 섹션 전에)
```

In [ ]:
import os, time, json, statistics as st
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
BASE = os.environ.get("LLM_BASE", "http://llm-serve:8000/v1")      # 도커 네트워크 gpu-lab 내부 주소
client = OpenAI(base_url=BASE, api_key=os.environ.get("LLM_API_KEY", "EMPTY"))
try:
    MODEL = client.models.list().data[0].id
    print("연결 OK →", BASE, "| 모델:", MODEL)
except Exception as e:
    print("서빙에 연결 실패:", e, "\n→ 서버에서 sudo bash scripts/17_serve.sh vllm 실행 후 다시 시도")

## 1. 스트리밍 + 지연 측정 [20]
**TTFT**(첫 토큰까지) 는 프리필 속도, **TPOT**(토큰 간 간격) 은 디코드 속도를 나타냅니다.

In [ ]:
def measure(messages, max_tokens=256, show=False, **kw):
    t0 = time.perf_counter(); first = None; n = 0; text = []
    stream = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0,
                                            stream=True, stream_options={"include_usage": True}, **kw)
    usage = None
    for ch in stream:
        if ch.usage: usage = ch.usage
        if ch.choices and ch.choices[0].delta.content:
            if first is None: first = time.perf_counter()
            text.append(ch.choices[0].delta.content); n += 1
            if show: print(ch.choices[0].delta.content, end="", flush=True)
    end = time.perf_counter()
    out_tok = usage.completion_tokens if usage else n
    r = dict(ttft_ms=(first - t0) * 1000, tpot_ms=(end - first) / max(out_tok - 1, 1) * 1000,
             tok_s=out_tok / (end - t0), out_tokens=out_tok)
    if show: print(f"\n\n→ TTFT {r['ttft_ms']:.0f} ms | TPOT {r['tpot_ms']:.1f} ms | {r['tok_s']:.1f} tok/s")
    return r, "".join(text)

_ = measure([{"role": "user", "content": "LLM 추론에서 프리필과 디코드 단계의 차이를 설명해 줘."}], show=True)

## 2. 동시 요청 — 연속 배칭(continuous batching) 효과 [20]
요청 8개를 동시에 보내도 개별 속도는 조금만 떨어지고 **총 처리량은 크게 오릅니다** (= 여러 학생이 동시에 써도 버팀).

In [ ]:
Q = [{"role": "user", "content": f"딥러닝 최적화 팁 {i+1}번을 한 문단으로 설명해 줘."} for i in range(8)]
for conc in (1, 8):
    t = time.perf_counter()
    with ThreadPoolExecutor(conc) as ex:
        rs = [r for r, _ in ex.map(lambda m: measure([m], max_tokens=200), Q)]
    wall = time.perf_counter() - t
    print(f"동시 {conc}: 총 {sum(r['out_tokens'] for r in rs)/wall:6.1f} tok/s | 요청당 {st.median(r['tok_s'] for r in rs):5.1f} tok/s"
          f" | TTFT p50 {st.median(r['ttft_ms'] for r in rs):5.0f} ms")

## 3. 엔진·기법 비교 실험 — Speculative Decoding [21] · FlashInfer [22] · AWQ/GPTQ [23]
1. 아래 셀을 실행해 현재 모드의 결과를 저장합니다 (`MODE_LABEL` 수정).
2. 서버에서 `17_serve.sh` 로 모드를 바꾸고 → 위의 **연결 셀** → 이 셀을 다시 실행합니다.
3. 마지막 셀이 모드별 결과를 그래프로 비교합니다.

**Speculative Decoding (n-gram)**: 출력이 입력 문맥을 반복하는 작업(코드 수정, 문서 교정, RAG 인용)에서 프롬프트 속 n-gram 으로 여러 토큰을 미리 추측하고 한 번에 검증 → **디코드 1.3~2배**. 자유 작문에서는 이득이 작습니다.

In [ ]:
MODE_LABEL = "vllm"      # ← 현재 서버 모드 이름으로 바꾸기 (vllm / vllm-spec / vllm-fi / vllm-gptq / sglang …)
CODE = "\n".join(f"    x{i} = layer{i}(x{i-1}).relu()" for i in range(1, 25))
tasks = {
    "자유 작문": [{"role": "user", "content": "GPU 서버 운영 경험담을 짧은 수필로 써 줘."}],
    "코드 수정(반복↑)": [{"role": "user", "content": f"아래 코드에서 relu() 를 gelu() 로 바꾼 전체 코드를 그대로 출력해. 코드만.\n\ndef f(x0):\n{CODE}\n    return x24"}],
}
row = {}
for name, msgs in tasks.items():
    rs = [measure(msgs, max_tokens=400)[0] for _ in range(3)]
    row[name] = {k: st.median(r[k] for r in rs) for k in ("ttft_ms", "tpot_ms", "tok_s")}
    print(f"[{MODE_LABEL}] {name:<12} {row[name]['tok_s']:6.1f} tok/s | TPOT {row[name]['tpot_ms']:5.1f} ms | TTFT {row[name]['ttft_ms']:5.0f} ms")
path = "serving_compare.json"
allr = json.load(open(path)) if os.path.exists(path) else {}
allr[MODE_LABEL] = row; json.dump(allr, open(path, "w"), ensure_ascii=False, indent=1)

In [ ]:
import matplotlib.pyplot as plt, numpy as np
allr = json.load(open("serving_compare.json"))
modes = list(allr); task_names = list(next(iter(allr.values())))
x = np.arange(len(modes)); w = 0.38
fig, ax = plt.subplots(figsize=(1.6 * len(modes) + 3, 4))
for i, tname in enumerate(task_names):
    ax.bar(x + (i - 0.5) * w, [allr[m].get(tname, {}).get("tok_s", 0) for m in modes], w,
           label=["free writing", "code edit (repetitive)"][i] if i < 2 else tname)
ax.set_xticks(x, modes); ax.set_ylabel("tokens/s per request"); ax.legend(); ax.set_title("Serving mode comparison")
plt.tight_layout(); plt.show()

## 4. Structured Generation — JSON · 선택지 · 도구 호출 [24]
제약 디코딩(xgrammar 등)이 **문법상 가능한 토큰만** 고르도록 막아서, 파싱 실패 재시도가 사라지고 에이전트 파이프라인이 빨라집니다.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal
class LabTask(BaseModel):
    title: str
    techniques: List[str] = Field(min_length=2, max_length=4)
    vram_gb: int
    difficulty: Literal["쉬움", "보통", "어려움"]

t = time.perf_counter()
r = client.chat.completions.create(
    model=MODEL, temperature=0, max_tokens=300,
    messages=[{"role": "user", "content": "12GB GPU 에서 할 만한 최적화 실습 과제 하나를 JSON 으로 제안해."}],
    response_format={"type": "json_schema", "json_schema": {"name": "lab_task", "schema": LabTask.model_json_schema(), "strict": True}})
task = LabTask.model_validate_json(r.choices[0].message.content)     # 스키마 검증 통과 보장
print(f"{(time.perf_counter()-t)*1000:.0f} ms →", task)

In [ ]:
# 선택지 강제 (분류기처럼 사용) — vLLM: guided_choice / SGLang 은 json_schema enum 으로 대체
try:
    r = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=5,
        messages=[{"role": "user", "content": "다음 문장의 감정은? '서버가 또 OOM 났어…'"}],
        extra_body={"guided_choice": ["긍정", "중립", "부정"]})
    print("guided_choice →", r.choices[0].message.content)
except Exception as e:
    print("이 엔진은 guided_choice 미지원:", str(e)[:100])

In [ ]:
# 도구 호출 (에이전트) — 17_serve.sh 가 --enable-auto-tool-choice / tool-call-parser 를 켜 둠
def gpu_status():
    return {"gpu": "RTX 4070 SUPER", "util_pct": 37, "vram_used_gb": 7.9, "power_w": 142, "power_limit_w": 180}
tools = [{"type": "function", "function": {"name": "gpu_status", "description": "실습 서버 GPU 현재 상태 조회",
                                           "parameters": {"type": "object", "properties": {}}}}]
msgs = [{"role": "user", "content": "지금 GPU 상태 어때? 전력 여유도 알려줘."}]
r = client.chat.completions.create(model=MODEL, messages=msgs, tools=tools, tool_choice="auto", temperature=0)
m = r.choices[0].message
if m.tool_calls:
    call = m.tool_calls[0]; print("도구 호출 →", call.function.name, call.function.arguments)
    msgs += [m.model_dump(exclude_none=True),
             {"role": "tool", "tool_call_id": call.id, "content": json.dumps(gpu_status(), ensure_ascii=False)}]
    r = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0)
    print("최종 답변 →", r.choices[0].message.content)
else:
    print("도구 호출 없이 답변:", m.content)

## 5. 노트북 안에서 vLLM 직접 실행 [8]
NGC PyTorch 를 깨지 않도록 vLLM 은 **별도 가상환경 커널**에 설치되어 있습니다.
1. 서빙 컨테이너를 끄거나(`17_serve.sh stop`) 아래 `gpu_memory_utilization` 을 낮추세요.
2. 상단 메뉴 **Kernel → Change Kernel → `vLLM (venv)`** 로 바꾼 뒤 이 섹션의 셀만 실행합니다.

In [ ]:
import sys
if "/opt/venvs/vllm" not in sys.prefix:
    print("현재 커널:", sys.prefix, "→ 'vLLM (venv)' 커널로 바꾼 뒤 실행하세요.")
else:
    from vllm import LLM, SamplingParams
    llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct", gpu_memory_utilization=0.45, max_model_len=4096)
    sp = SamplingParams(temperature=0, max_tokens=128)
    prompts = [f"{t}를 한 문장으로 설명해 줘." for t in ["KV 캐시", "PagedAttention", "연속 배칭", "Speculative Decoding"]]
    t = time.perf_counter()
    outs = llm.chat([[{"role": "user", "content": p}] for p in prompts], sp)
    dt = time.perf_counter() - t
    for o in outs:
        print("·", o.outputs[0].text.strip()[:120])
    print(f"\n오프라인 배치 {len(prompts)}개: {sum(len(o.outputs[0].token_ids) for o in outs)/dt:.0f} tok/s")

## 6. Torch-TensorRT — CNN 추론 컴파일 [34]
> 이 섹션부터는 **기본 커널(Python 3)** 에서, 서빙을 끈 상태(`17_serve.sh stop`)로 실행하세요.

NGC 컨테이너에 내장된 `torch_tensorrt` 로 ResNet-50 을 TensorRT 엔진으로 컴파일해 eager / torch.compile 과 비교합니다.

In [ ]:
import torch, torchvision
dev = "cuda"
free = torch.cuda.mem_get_info()[0] / 2**30
print(f"VRAM 여유 {free:.1f} GB", "⚠️ 서빙을 끄세요" if free < 4 else "")
def bench(fn, iters=50, warmup=10):
    for _ in range(warmup): fn()
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(iters): fn()
    torch.cuda.synchronize(); return (time.perf_counter() - t) / iters

B = 32
model = torchvision.models.resnet50(num_classes=1000).eval().to(dev)
x = torch.randn(B, 3, 224, 224, device=dev)
res = {}
with torch.no_grad():
    res["eager FP32(TF32)"] = bench(lambda: model(x))
    mh, xh = model.half(), x.half()
    res["eager FP16"] = bench(lambda: mh(xh))
    mc = torch.compile(mh, mode="max-autotune-no-cudagraphs")
    res["torch.compile FP16"] = bench(lambda: mc(xh))
try:
    import torch_tensorrt
    trt_model = torch_tensorrt.compile(mh, ir="dynamo", inputs=[xh], enabled_precisions={torch.half})
    with torch.no_grad():
        res["Torch-TensorRT FP16"] = bench(lambda: trt_model(xh))
    diff = (trt_model(xh).float() - mh(xh).float()).abs().max().item()
    print(f"torch_tensorrt {torch_tensorrt.__version__} | 최대 오차 {diff:.3e}")
except Exception as e:
    print("Torch-TensorRT 실패:", str(e)[:200])
for k, v in res.items():
    print(f"{k:<22} {v*1000:7.2f} ms/배치  {B/v:7.0f} img/s")

## 7. ONNX Runtime + TensorRT EP [35]
프레임워크 독립 배포 표준 경로: PyTorch → ONNX → ONNX Runtime(TensorRT/CUDA 실행 공급자).

In [ ]:
import onnxruntime as ort, numpy as np
os.makedirs("onnx", exist_ok=True)
torch.onnx.export(model.float(), x[:1].float(), "onnx/resnet50.onnx", opset_version=17,
                  input_names=["input"], output_names=["logits"], dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}})
print("ORT", ort.__version__, "| 사용 가능 EP:", ort.get_available_providers())
xn = x.float().cpu().numpy()
for name, providers in [
    ("ORT TensorRT EP FP16", [("TensorrtExecutionProvider", {"trt_fp16_enable": True, "trt_engine_cache_enable": True,
                                                              "trt_engine_cache_path": "onnx/trt_cache"}), "CUDAExecutionProvider"]),
    ("ORT CUDA EP", ["CUDAExecutionProvider"])]:
    try:
        sess = ort.InferenceSession("onnx/resnet50.onnx", providers=providers)
        used = sess.get_providers()[0]
        sess.run(None, {"input": xn})                                     # 엔진 빌드/워밍업 (TRT 는 수십 초)
        t = time.perf_counter()
        for _ in range(20): sess.run(None, {"input": xn})
        dt = (time.perf_counter() - t) / 20
        print(f"{name:<22} 실제 EP={used:<28} {dt*1000:7.2f} ms/배치  {B/dt:7.0f} img/s")
    except Exception as e:
        print(f"{name:<22} 실패: {str(e)[:150]}")
print("※ ORT 는 입력을 CPU numpy 로 넘겨 H2D 복사 시간이 포함됩니다 (IOBinding 으로 제거 가능).")

In [ ]:
del model, mh, mc
try: del trt_model
except NameError: pass
import gc; gc.collect(); torch.cuda.empty_cache()
print(f"VRAM 정리 완료: {torch.cuda.memory_allocated()/2**30:.2f} GB 사용")